# Compare Nano vs Base vs Large

WorldKit ships with four model configs: `nano`, `base`, `large`, and `xl`.
Bigger models have more capacity but cost more compute. This notebook
trains all three smaller configs on the same data and compares them
head-to-head on prediction quality, speed, and latent space quality.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DilpreetBansi/worldkit/blob/main/notebooks/05_model_comparison.ipynb)

**Time to run:** ~10 minutes on Colab T4

In [ ]:
!pip install "worldkit[train]" -q

## 1. Model Configurations

Let's look at what each config gives us before training.

In [ ]:
from worldkit import get_config

configs = {}
for name in ["nano", "base", "large"]:
    cfg = get_config(name, action_dim=2)
    configs[name] = cfg
    print(f"\n--- {name.upper()} ---")
    print(f"  Latent dim:     {cfg.latent_dim}")
    print(f"  Predictor:      depth={cfg.pred_depth}, heads={cfg.pred_heads}")
    print(f"  Image size:     {cfg.image_size}")
    print(f"  Encoder embed:  {cfg.encoder_embed_dim}")

## 2. Create Training and Evaluation Data

We'll use synthetic bouncing-ball data so the results are reproducible
on any machine. Same data for all models — fair comparison.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import time


def make_bouncing_ball(n_episodes, timesteps, size=96, seed=0):
    """Generate bouncing ball episodes."""
    rng = np.random.RandomState(seed)
    all_pix = []
    all_act = []
    for _ in range(n_episodes):
        x = rng.uniform(12, size - 12)
        y = rng.uniform(12, size - 12)
        vx = rng.uniform(2, 5) * rng.choice([-1, 1])
        vy = rng.uniform(2, 5) * rng.choice([-1, 1])
        frames = []
        for _ in range(timesteps):
            frame = np.full((size, size, 3), 30, dtype=np.uint8)
            yy, xx = np.ogrid[:size, :size]
            mask = (xx - int(x)) ** 2 + (yy - int(y)) ** 2 <= 64
            frame[mask] = [70, 130, 230]
            frames.append(frame)
            x += vx; y += vy
            if x < 8 or x > size - 8: vx = -vx; x = np.clip(x, 8, size - 8)
            if y < 8 or y > size - 8: vy = -vy; y = np.clip(y, 8, size - 8)
        all_pix.append(np.array(frames))
        all_act.append(np.zeros((timesteps, 2), dtype=np.float32))
    return np.array(all_pix, dtype=np.uint8), np.array(all_act)


# Training data
train_pix, train_act = make_bouncing_ball(300, 32, seed=0)
with h5py.File("compare_train.h5", "w") as f:
    f.create_dataset("pixels", data=train_pix, compression="gzip")
    f.create_dataset("actions", data=train_act, compression="gzip")

# Evaluation data (different seed)
eval_pix, eval_act = make_bouncing_ball(50, 32, seed=999)
with h5py.File("compare_eval.h5", "w") as f:
    f.create_dataset("pixels", data=eval_pix, compression="gzip")
    f.create_dataset("actions", data=eval_act, compression="gzip")

print(f"Training:   {train_pix.shape}")
print(f"Evaluation: {eval_pix.shape}")

## 3. Train All Three Models

Same data, same epochs, same hyperparameters — only the model size changes.

In [ ]:
from worldkit import WorldModel

models = {}
train_times = {}

for config_name in ["nano", "base", "large"]:
    print(f"\n{'='*50}")
    print(f"Training {config_name.upper()}...")
    print(f"{'='*50}")
    start = time.time()

    model = WorldModel.train(
        data="compare_train.h5",
        config=config_name,
        epochs=15,
        batch_size=32,
        action_dim=2,
        device="auto",
    )

    elapsed = time.time() - start
    models[config_name] = model
    train_times[config_name] = elapsed
    print(f"{config_name}: {model.num_params:,} params, trained in {elapsed:.1f}s")

## 4. Compare Prediction Quality

For each model, predict future states on the evaluation data and
measure prediction confidence.

In [ ]:
# Evaluate each model on the eval set
results = {}

for name, model in models.items():
    confidences = []
    encode_times = []
    predict_times = []

    for ep in range(min(30, eval_pix.shape[0])):
        frame = eval_pix[ep, 0]
        actions = [(0.0, 0.0)] * 10

        # Time encoding
        t0 = time.time()
        model.encode(frame)
        encode_times.append((time.time() - t0) * 1000)

        # Time prediction
        t0 = time.time()
        pred = model.predict(frame, actions=actions)
        predict_times.append((time.time() - t0) * 1000)
        confidences.append(pred.confidence)

    results[name] = {
        "params": model.num_params,
        "latent_dim": model.latent_dim,
        "train_time": train_times[name],
        "mean_confidence": np.mean(confidences),
        "mean_encode_ms": np.mean(encode_times),
        "mean_predict_ms": np.mean(predict_times),
    }

# Print comparison table
print(f"{'Metric':<25} {'Nano':>12} {'Base':>12} {'Large':>12}")
print("-" * 61)
for metric in ["params", "latent_dim", "train_time", "mean_confidence",
               "mean_encode_ms", "mean_predict_ms"]:
    vals = [results[n][metric] for n in ["nano", "base", "large"]]
    if metric == "params":
        print(f"{metric:<25} {vals[0]:>12,} {vals[1]:>12,} {vals[2]:>12,}")
    elif metric == "train_time":
        print(f"{metric + ' (s)':<25} {vals[0]:>12.1f} {vals[1]:>12.1f} {vals[2]:>12.1f}")
    else:
        print(f"{metric:<25} {vals[0]:>12.4f} {vals[1]:>12.4f} {vals[2]:>12.4f}")

In [ ]:
# Visualize the comparison
names = ["nano", "base", "large"]
colors = ["steelblue", "coral", "mediumseagreen"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Parameters
params = [results[n]["params"] / 1e6 for n in names]
axes[0].bar(names, params, color=colors)
axes[0].set_ylabel("Parameters (M)")
axes[0].set_title("Model size")

# Training time
ttimes = [results[n]["train_time"] for n in names]
axes[1].bar(names, ttimes, color=colors)
axes[1].set_ylabel("Time (seconds)")
axes[1].set_title("Training time")

# Prediction confidence
confs = [results[n]["mean_confidence"] for n in names]
axes[2].bar(names, confs, color=colors)
axes[2].set_ylabel("Mean confidence")
axes[2].set_title("Prediction confidence")

# Inference speed
enc_ms = [results[n]["mean_encode_ms"] for n in names]
pred_ms = [results[n]["mean_predict_ms"] for n in names]
x = np.arange(len(names))
w = 0.35
axes[3].bar(x - w/2, enc_ms, w, label="Encode", color="steelblue")
axes[3].bar(x + w/2, pred_ms, w, label="Predict", color="coral")
axes[3].set_xticks(x)
axes[3].set_xticklabels(names)
axes[3].set_ylabel("Time (ms)")
axes[3].set_title("Inference latency")
axes[3].legend()

plt.tight_layout()
plt.show()

## 5. Plausibility Score Comparison

Compare how well each model distinguishes real from random sequences.

In [ ]:
n_test = 30

plaus_results = {}
for name, model in models.items():
    real_scores = []
    random_scores = []
    for i in range(n_test):
        # Real sequence from eval data
        real_seq = list(eval_pix[i, :8])
        real_scores.append(model.plausibility(real_seq))
        # Random sequence
        rand_seq = [np.random.randint(0, 255, (96, 96, 3), dtype=np.uint8) for _ in range(8)]
        random_scores.append(model.plausibility(rand_seq))
    plaus_results[name] = {"real": real_scores, "random": random_scores}

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, names):
    ax.hist(plaus_results[name]["real"], bins=12, alpha=0.7,
            color="steelblue", label="Real")
    ax.hist(plaus_results[name]["random"], bins=12, alpha=0.7,
            color="coral", label="Random")
    ax.set_title(f"{name.upper()}")
    ax.set_xlabel("Plausibility score")
    ax.set_ylabel("Count")
    ax.legend()

    # Print separation
    gap = np.mean(plaus_results[name]["real"]) - np.mean(plaus_results[name]["random"])
    print(f"{name}: real={np.mean(plaus_results[name]['real']):.4f}, "
          f"random={np.mean(plaus_results[name]['random']):.4f}, gap={gap:.4f}")

plt.suptitle("Plausibility: real vs random sequences", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Latent Space Quality

Compare the structure of each model's latent space by looking at
how latent distances correlate with visual similarity.

In [ ]:
from sklearn.decomposition import PCA

# Encode a shared set of frames with each model
shared_frames = eval_pix[:20].reshape(-1, 96, 96, 3)[:200]  # 200 frames

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, name in zip(axes, names):
    model = models[name]
    latents = np.array([model.encode(f) for f in shared_frames])

    pca = PCA(n_components=2)
    z2d = pca.fit_transform(latents)

    # Color by frame index (temporal order)
    sc = ax.scatter(z2d[:, 0], z2d[:, 1], c=np.arange(len(z2d)),
                    cmap="viridis", s=10, alpha=0.7)
    ax.set_title(f"{name.upper()} (PCA)\nvar: {pca.explained_variance_ratio_.sum():.1%}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    plt.colorbar(sc, ax=ax, label="Frame index")

plt.suptitle("Latent space structure (colored by time)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. When to Use Which Config

| Config | Parameters | Best for |
|--------|-----------|----------|
| **nano** | ~5M | Prototyping, Colab, edge deployment, simple environments |
| **base** | ~15M | Production use, most environments, good quality/speed tradeoff |
| **large** | ~50M | Complex environments, high-fidelity predictions, research |
| **xl** | ~100M | State-of-the-art, multi-environment, large-scale training |

**Rules of thumb:**
- Start with `nano` to validate your pipeline
- Move to `base` for real results
- Use `large`/`xl` only if `base` isn't good enough and you have the data

## Next Steps

| Notebook | What's next |
|----------|-------------|
| [04 — Latent Probing](04_latent_probing.ipynb) | Probe what each model size learns |
| [06 — Export & Deploy](06_export_deploy.ipynb) | Export the best model to ONNX |
| [07 — Plan Robot Actions](07_plan_robot_actions.ipynb) | Compare planning quality across sizes |